In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from functools import cache

In [5]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obvs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides

# Run Sim

In [8]:
config = EnvConfig(max_episode_steps=60 * 3)
plt_cfg = PlotConfig()

In [9]:
GAMMA = get_gamma_from_half_life(config.max_episode_steps // 2)
round(GAMMA, 5)

0.99233

In [10]:
def discounted_rewards(rewards: np.ndarray, gamma: float = 0.997):
    """
    Discount rewards using discount factor gamma.
    """
    disc_schedule = exp_decay_half_life(len(rewards), gamma=gamma)
    return rewards * disc_schedule


def get_act_dict(agents: tuple[PricingAgent, DispatchAgent, RepositionAgent], obs: ObservationDict) -> ActionDict:
    price_agent, dispatch_agent, reposition_agent = agents
    prices = price_agent.price(obs)
    dispatch_actions = dispatch_agent.dispatch(obs)
    reposition_actions = reposition_agent.reposition(obs)

    action_agent: ActionDict = {
        "prices": prices,
        "dispatch": dispatch_actions,
        "reposition": reposition_actions,
    }
    return action_agent

In [11]:
def run_heuristic_simulation(env_curr: RideShareEnv, num_iter: int = 1):
    REWARDS: list[list[float]] = []
    OBS: list[list[ObservationDict]] = []
    ACT: list[list[ActionDict]] = []

    agents = (PricingAgent(env_curr.config), DispatchAgent(env_curr), RepositionAgent(env_curr))

    tqdm.write("Starting heuristic simulation...")

    for iter_count in tqdm(range(num_iter)):
        rews_raw: list[float] = []
        obs_list: list[ObservationDict] = []
        act_list: list[ActionDict] = []
        done = False
        obs, info = env_curr.reset()

        while not done:
            obs: ObservationDict
            obs_list.append(obs)

            action_agent: ActionDict = get_act_dict(agents, obs)
            act_list.append(action_agent)

            obs, reward, term, done, info = env_curr.step(action_agent)  # type: ignore

            rews_raw.append(reward)
            if term:
                tqdm.write(f"WARNING: Episode terminated at step {env_curr.current_step}")
            done = done or term

        rews = discounted_rewards(np.array(rews_raw), gamma=GAMMA).tolist()
        REWARDS.append(rews)
        OBS.append(obs_list)
        ACT.append(act_list)
    return (REWARDS), OBS, ACT, env_curr, agents

In [12]:
env_heuristic = RideShareEnv(config, plt_cfg)
G = env_heuristic.G
cfg = env_heuristic.config

Assigned lambda values to nodes. Total lambda: 2.2856 (target: 2.3460)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [13]:
REWARDS, OBS, ACT, env_curr, agents = run_heuristic_simulation(env_heuristic, num_iter=1)

Starting heuristic simulation...


100%|██████████| 1/1 [00:11<00:00, 11.46s/it]


In [14]:
arr = np.array(REWARDS)
arr.shape

(1, 181)

In [15]:
arr2 = arr.reshape(-1, arr.shape[0])
arr2.shape

(181, 1)

In [16]:
pd.DataFrame(arr2).head(2)

,0
0,0.0
1,0.0


In [17]:
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import (
    train_ppo,
    PPOTrainConfig,
    RideShareActorCritic,
    obs_numpy_to_torch,
    action_torch_to_numpy,
)

In [18]:
env_model = RideShareEnv(config, plt_cfg)
model = RideShareActorCritic(env_model)

Assigned lambda values to nodes. Total lambda: 2.2255 (target: 2.3460)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [19]:
obs, info = env_model.reset()

In [20]:
veh, req, rides = get_obvs_tuple(env_model)

In [21]:
@torch.no_grad()
def run_model_simulation(
    env_curr: RideShareEnv,
    model: RideShareActorCritic,
    num_iter: int = 1,
    gamma: float = GAMMA,
    deterministic: bool = False,
) -> tuple[list[list[float]], list[list[dict[str, np.ndarray]]], list[list[dict[str, np.ndarray]]]]:
    """
    Returns:
      REWARDS: list[episode][t] discounted reward_t
      OBS:     list[episode][t] obs dict (numpy)
      ACT:     list[episode][t] action dict (numpy)
    """
    model.eval()

    REWARDS: list[list[float]] = []
    OBS: list[list[dict[str, np.ndarray]]] = []
    ACT: list[list[dict[str, np.ndarray]]] = []

    for _ in tqdm(range(num_iter), desc="model rollout"):
        obs_np, _info = env_curr.reset()
        done = False

        rews_raw: list[float] = []
        obs_list: list[dict[str, np.ndarray]] = []
        act_list: list[dict[str, np.ndarray]] = []

        while not done:
            obs_list.append(obs_np)

            obs_t = obs_numpy_to_torch(obs_np)
            act_t = model.act(obs_t, deterministic=deterministic)
            act_np = action_torch_to_numpy(act_t)
            act_list.append(act_np)

            obs_np, reward, terminated, truncated, _info = env_curr.step(act_np)  # type: ignore[arg-type]
            rews_raw.append(float(reward))
            done = bool(terminated) or bool(truncated)

        rews = discounted_rewards(np.array(rews_raw, dtype=np.float32), gamma=gamma).tolist()
        REWARDS.append(t.cast(list[float], rews))
        OBS.append(obs_list)
        ACT.append(act_list)

    return REWARDS, OBS, ACT

In [22]:
run_model_simulation(env_model, model, num_iter=1)

model rollout:   0%|          | 0/1 [00:00<?, ?it/s]/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/models/ppo_model.py:160: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  price_sigma = out["price_mu"].new_tensor(self.price_logstd).exp()
/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/models/ppo_model.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  repo_sigma = out["repo_mu"].new_tensor(self.repo_logstd).exp().view(self.num_veh, 2)
model rollout: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it]


([[0.0]],
 [[{'globals': array([-0.7818,  0.6235,  0.6412,  0.7674,  1.    ]),
    'supply_demand_ratio': array([0.99998227, 0.03      , 0.003     ]),
    'vehicles':     loc_x_norm  loc_y_norm   battery  status
    0     0.064259    0.663667  0.781625       0
    1     0.053067    0.167395  0.914796       0
    2    -0.122785   -0.607159  0.735104       0
    3     0.245935    0.252984  0.701260       0
    4    -0.238117    0.223407  0.813246       0
    5     0.078044   -0.025294  0.951773       0
    6    -0.261076   -0.024656  0.790087       0
    7    -0.447019   -0.516556  0.639364       0
    8     0.080418   -0.129867  0.624128       0
    9     0.190569    0.085253  0.895381       0
    10   -0.504266   -0.510795  0.760559       0
    11   -0.332132   -0.042836  0.667575       0
    12    0.058644   -0.159496  0.774182       0
    13   -0.094700    0.396370  0.744626       0
    14   -0.202490   -0.484012  0.993715       0
    15    0.058644   -0.159496  0.852099       0
    

In [23]:
obs["pending_requests"]

,pickup_x_norm,pickup_y_norm,dropoff_x_norm,dropoff_y_norm,distance_meters,cust_bias,cust_temperature,est_cost,max_wait_time,wait_time,status
0,0.218219,0.711217,0.038168,0.146179,4639.728582,2.612326,2.471162,4.639729,0 days 00:15:00,0 days,0
1,0.248831,-0.028007,0.318062,0.394210,4025.683132,0.667547,2.418839,4.025683,0 days 00:15:00,0 days,0
2,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
3,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
4,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
5,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
6,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
7,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
8,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10
9,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0 days 00:15:00,0 days,-10


In [24]:
env = RideShareEnv()
cfg = PPOTrainConfig(total_steps=5, rollout_len=3)
model, logs = train_ppo(env, cfg=cfg)

Assigned lambda values to nodes. Total lambda: 2.3208 (target: 2.3460)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/models/ppo_model.py:160: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  price_sigma = out["price_mu"].new_tensor(self.price_logstd).exp()
/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/models/ppo_model.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().require

ValueError: Expected value argument (Tensor of shape (50,)) to be within the support (GreaterThan(lower_bound=0.0)) of the distribution LogNormal(), but found invalid values:
tensor([0., 0., inf, 0., 0., 0., nan, 0., nan, 0., 0., nan, nan, 0., 0., 0., nan, 0., nan, nan, nan, 0., nan, 0.,
        nan, nan, 0., 0., 0., 0., nan, 0., nan, nan, nan, 0., nan, 0., 0., nan, 0., 0., nan, 0., 0., nan, nan, nan,
        nan, 0.], device='mps:0')

In [ ]:
# env.breadcrumbs["rewards"].cumsum().plot(title="Cumulative Reward over Time")
# plt.xlabel("Time Step")
# plt.ylabel("Cumulative Reward")
# plt.show()

In [ ]:
# fig, ax = env.render()
# fig